In [ ]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, f1_score, confusion_matrix
import joblib
import random

#Step 1: Objective of the model
To classify Ghanaian recipes as safe or unsafe for users with specific health conditions (diabetes, hypertension, cholesterol-related conditions)

# Step 2: Data Preprocessing

In [ ]:
data = pd.read_csv("ghana_recipes_v2.csv")

In [ ]:
data.head()

,recipe_id,name,ingredients,instructions,sodium_mg,sugar_g,fiber_g,fat_total_g,fat_saturated_g,potassium_mg,iron_mg,meal_type,tags,is_diabetic_safe,is_hypertensive_safe,is_cholesterol_safe,image_url
0,1,Red Red (Bean Stew),Black-eyed peas; Palm oil; Onion; Plantain,Soak beans; Boil till soft; Fry plantain in oi...,580,18.0,14.5,24.0,9.5,850,5.2,Lunch,Vegetarian High-Fiber,0,0,0,https://placehold.co/400x300?text=Red+Red+(Bea...
1,2,Banku and Grilled Tilapia,Corn dough; Cassava dough; Tilapia; Pepper,Mix doughs with water; Cook vigorously; Grill ...,420,2.0,3.5,13.2,3.0,450,2.1,Dinner,High-Protein Gluten-Free,1,1,1,https://placehold.co/400x300?text=Banku+and+Gr...
2,3,Fufu and Goat Light Soup,Cassava; Plantain; Goat meat; Garden eggs,Boil cassava and plantain; Pound together; Boi...,650,5.0,4.0,18.0,6.5,600,3.8,Lunch,Traditional High-Calorie,1,0,0,https://placehold.co/400x300?text=Fufu+and+Goa...
3,4,Garden Egg Stew with Yam,Garden eggs; Palm oil; Koobi; Yam,Boil garden eggs and mash; Fry onions and koob...,290,6.0,9.2,14.0,5.5,720,1.8,Lunch,Low-Sodium Vegetable-Rich,1,1,0,https://placehold.co/400x300?text=Garden+Egg+S...
4,5,Waakye with Shito,Rice; Black-eyed peas; Sorghum leaves; Oil,Boil beans with sorghum; Add rice; Cook togeth...,480,3.0,11.0,16.0,4.0,510,4.5,Breakfast,High-Fiber Street-Food,1,1,1,https://placehold.co/400x300?text=Waakye+with+...


In [ ]:
print(data.dtypes)

recipe_id                 int64
name                     object
ingredients              object
instructions             object
sodium_mg                 int64
sugar_g                 float64
fiber_g                 float64
fat_total_g             float64
fat_saturated_g         float64
potassium_mg              int64
iron_mg                 float64
meal_type                object
tags                     object
is_diabetic_safe          int64
is_hypertensive_safe      int64
is_cholesterol_safe       int64
image_url                object
dtype: object


In [ ]:
print(data.columns)

Index(['recipe_id', 'name', 'ingredients', 'instructions', 'sodium_mg',
       'sugar_g', 'fiber_g', 'fat_total_g', 'fat_saturated_g', 'potassium_mg',
       'iron_mg', 'meal_type', 'tags', 'is_diabetic_safe',
       'is_hypertensive_safe', 'is_cholesterol_safe', 'image_url'],
      dtype='object')


In [ ]:
x = data[['sodium_mg', 'sugar_g', 'fiber_g',
         'fat_total_g', 'fat_saturated_g', 'potassium_mg', 'iron_mg'
         , 'meal_type', 'tags']]

In [ ]:
data_categorical = x.select_dtypes(include=['object'])

In [ ]:
encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
data_categorical_encoded = encoder.fit_transform(data_categorical)
encoded_feature_names = encoder.get_feature_names_out(data_categorical.columns)

In [ ]:
data_encoded_df = pd.DataFrame(
    data_categorical_encoded, columns=encoded_feature_names, index=x.index)

In [ ]:
data_numerical = x.drop(columns=data_categorical.columns)

In [ ]:
x_final = pd.concat([data_numerical, data_encoded_df], axis = 1)

In [ ]:
x_final

,sodium_mg,sugar_g,fiber_g,fat_total_g,fat_saturated_g,potassium_mg,iron_mg,meal_type_Breakfast,meal_type_Dessert,meal_type_Dinner,...,tags_Quick-Meal High-Sodium,tags_Quick-Meal High-Sugar,tags_Quick-Meal Student-Favorite,tags_Street-Food Vegan,tags_Traditional Fermented,tags_Traditional High-Calorie,tags_Traditional High-Fat,tags_Traditional Nutty,tags_Traditional Spicy,tags_Vegetarian High-Fiber
0,580,18.0,14.5,24.0,9.5,850,5.2,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,420,2.0,3.5,13.2,3.0,450,2.1,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,650,5.0,4.0,18.0,6.5,600,3.8,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
3,290,6.0,9.2,14.0,5.5,720,1.8,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,480,3.0,11.0,16.0,4.0,510,4.5,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,150,1.0,6.5,8.0,1.5,300,5.5,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6,410,1.5,4.0,22.0,8.0,380,2.5,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
7,450,4.0,2.5,28.0,8.0,600,3.5,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8,320,12.0,8.5,18.5,8.0,950,6.5,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9,550,4.5,3.0,14.0,3.5,480,2.8,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


# Step 3: Training the model

In [ ]:
def train_health_dt_model(target_column): # add model file name later
  y = data[target_column]
  X_train, X_test, y_train, y_test = train_test_split(x_final, y, test_size=0.2, random_state=42, stratify=y)

  dt_model = DecisionTreeClassifier(random_state=42)
  dt_model.fit(X_train, y_train)

  y_pred = dt_model.predict(X_test)

  dt_accuracy = accuracy_score(y_test, y_pred)
  dt_precision = precision_score(y_test, y_pred)
  dt_f1_score = f1_score(y_test, y_pred)
  dt_cm = confusion_matrix(y_test, y_pred)

  print(f"Accuracy: {dt_accuracy}")
  print(f"Precision: {dt_precision}")
  print(f"F1 Score: {dt_f1_score}")
  print(f"Confusion Matrix:\n{dt_cm}")

  return dt_model




In [ ]:
dt_diabetes_model = train_health_dt_model("is_diabetic_safe")
dt_diabetes_model

Accuracy: 0.9
Precision: 0.875
F1 Score: 0.9333333333333333
Confusion Matrix:
[[2 1]
 [0 7]]


DecisionTreeClassifier(random_state=42)

In [ ]:
dt_hypertensive_model = train_health_dt_model("is_hypertensive_safe")
dt_hypertensive_model

Accuracy: 1.0
Precision: 1.0
F1 Score: 1.0
Confusion Matrix:
[[2 0]
 [0 8]]


DecisionTreeClassifier(random_state=42)

In [ ]:
dt_cholestrol_model = train_health_dt_model("is_cholesterol_safe")
dt_cholestrol_model

Accuracy: 0.9
Precision: 0.8333333333333334
F1 Score: 0.9090909090909091
Confusion Matrix:
[[4 1]
 [0 5]]


DecisionTreeClassifier(random_state=42)

In [ ]:
def train_health_rf_model(target_column): # add model file name later
  y = data[target_column]
  X_train, X_test, y_train, y_test = train_test_split(x_final, y, test_size=0.2, random_state=42, stratify=y)

  rf_model = RandomForestClassifier(random_state=42)
  rf_model.fit(X_train, y_train)

  y_pred = rf_model.predict(X_test)

  rf_accuracy = accuracy_score(y_test, y_pred)
  rf_precision = precision_score(y_test, y_pred)
  rf_f1_score = f1_score(y_test, y_pred)
  rf_cm = confusion_matrix(y_test, y_pred)

  print(f"Accuracy: {rf_accuracy}")
  print(f"Precision: {rf_precision}")
  print(f"F1 Score: {rf_f1_score}")
  print(f"Confusion Matrix:\n{rf_cm}")

  return rf_model



In [ ]:
rf_diabetes_model = train_health_rf_model("is_diabetic_safe")
rf_diabetes_model

Accuracy: 0.9
Precision: 0.875
F1 Score: 0.9333333333333333
Confusion Matrix:
[[2 1]
 [0 7]]


RandomForestClassifier(random_state=42)

In [ ]:
rf_hypertensive_model = train_health_rf_model("is_hypertensive_safe")
rf_hypertensive_model

Accuracy: 0.9
Precision: 0.8888888888888888
F1 Score: 0.9411764705882353
Confusion Matrix:
[[1 1]
 [0 8]]


RandomForestClassifier(random_state=42)

In [ ]:
rf_cholestrol_model = train_health_rf_model("is_cholesterol_safe")
rf_cholestrol_model

Accuracy: 1.0
Precision: 1.0
F1 Score: 1.0
Confusion Matrix:
[[5 0]
 [0 5]]


RandomForestClassifier(random_state=42)

In [ ]:
def train_health_gb_model(target_column): # add model file name later
  y = data[target_column]
  X_train, X_test, y_train, y_test = train_test_split(x_final, y, test_size=0.2, random_state=42, stratify=y)

  gb_model = GradientBoostingClassifier(random_state=42)
  gb_model.fit(X_train, y_train)

  y_pred = gb_model.predict(X_test)

  gb_accuracy = accuracy_score(y_test, y_pred)
  gb_precision = precision_score(y_test, y_pred)
  gb_f1_score = f1_score(y_test, y_pred)
  gb_cm = confusion_matrix(y_test, y_pred)

  print(f"Accuracy: {gb_accuracy}")
  print(f"Precision: {gb_precision}")
  print(f"F1 Score: {gb_f1_score}")
  print(f"Confusion Matrix:\n{gb_cm}")

  return gb_model




In [ ]:
gb_diabetes_model = train_health_gb_model('is_diabetic_safe')
gb_diabetes_model

Accuracy: 0.9
Precision: 0.875
F1 Score: 0.9333333333333333
Confusion Matrix:
[[2 1]
 [0 7]]


GradientBoostingClassifier(random_state=42)

In [ ]:
gb_cholestrol_model = train_health_gb_model('is_cholesterol_safe')
gb_cholestrol_model

Accuracy: 1.0
Precision: 1.0
F1 Score: 1.0
Confusion Matrix:
[[5 0]
 [0 5]]


GradientBoostingClassifier(random_state=42)

In [ ]:
gb_hypertensive_model = train_health_gb_model('is_hypertensive_safe')
gb_hypertensive_model

Accuracy: 1.0
Precision: 1.0
F1 Score: 1.0
Confusion Matrix:
[[2 0]
 [0 8]]


GradientBoostingClassifier(random_state=42)

In [ ]:
final_diabetes_model = dt_diabetes_model
final_hypertension_model = dt_hypertensive_model
final_cholestrol_model = dt_cholestrol_model

joblib.dump(final_diabetes_model, 'diabetes_model.pkl')
joblib.dump(final_hypertension_model, 'hypertensive_model.pkl')
joblib.dump(final_cholestrol_model, 'cholestrol_model.pkl')

joblib.dump(encoder, 'encoder.pkl')
joblib.dump(x_final.columns, 'model_columns.pkl')

['model_columns.pkl']

#Testing the Model

In [ ]:
models = {
    "Diabetes": joblib.load('diabetes_model.pkl'),
    "Hypertensive": joblib.load('hypertensive_model.pkl'),
    "Cholestrol": joblib.load('cholestrol_model.pkl')
}

encoder = joblib.load('encoder.pkl')
model_columns = joblib.load('model_columns.pkl')

In [ ]:
TRAINING_FEATURES = [
    'sodium_mg', 'sugar_g', 'fiber_g', 'fat_total_g',
    'fat_saturated_g', 'potassium_mg', 'iron_mg',
    'meal_type', 'tags'
]

Prompt:
Yeah, I meant I want to test the model to see if it would give correct recipes based on the health condition and if the user wants lunch breakfast and so on. Show me how to do that step by step

AI: Gemini 3 Pro


In [ ]:
def get_dynamic_recommendation(health_condition, meal_request, dataset, current_reading=None):
    """
    current_reading: The user's level TODAY.
       - For Hypertension: Current BP (e.g., 140)
       - For Diabetes: Current Sugar (e.g., 180)
    """
    print(f"\n🔍 REQUEST: '{meal_request}' for '{health_condition}'")

    # 1. ESTABLISH "STRICT MODE" (The Tracking Feature)
    strict_mode = False

    if health_condition == "Hypertension" and current_reading is not None:
        if current_reading >= 140: # High BP threshold
            print(f"⚠️ ALERT: High BP ({current_reading}) detected! Activating STRICT LOW-SODIUM MODE.")
            strict_mode = True

    if health_condition == "Diabetes" and current_reading is not None:
        if current_reading >= 180: # High Sugar threshold
            print(f"⚠️ ALERT: High Sugar ({current_reading}) detected! Activating ZERO-SUGAR MODE.")
            strict_mode = True

    # 2. STANDARD AI PREDICTION (Your existing logic)
    # (Assuming you loaded models/encoders/columns globally as before)
    active_model = models[health_condition]

    # --- Data Prep Pipeline (Same as before) ---
    df_clean = dataset[TRAINING_FEATURES].copy()
    df_num = df_clean.select_dtypes(exclude=['object'])
    df_cat = df_clean.select_dtypes(include=['object'])

    try:
        encoded_matrix = encoder.transform(df_cat)
        df_encoded = pd.DataFrame(encoded_matrix, columns=encoder.get_feature_names_out(df_cat.columns), index=df_clean.index)
        df_input = pd.concat([df_num, df_encoded], axis=1)
        df_input = df_input.reindex(columns=model_columns, fill_value=0)

        # Predict
        dataset['is_safe_ai'] = active_model.predict(df_input)
    except Exception as e:
        print(f"Pipeline Error: {e}")
        return

    # 3. FILTERING (AI + Meal Type)
    safe_pool = dataset[
        (dataset['is_safe_ai'] == 1) &
        (dataset['meal_type'] == meal_request)
    ]

    # 4. APPLY "STRICT MODE" (The Tracker Logic)
    final_suggestions = []

    for idx, row in safe_pool.iterrows():
        # A. If Strict Mode is ON, apply extra hard rules
        if strict_mode:
            if health_condition == "Hypertension" and row['sodium_mg'] > 200: # Super low salt only
                continue
            if health_condition == "Diabetes" and row['sugar_g'] > 5: # Super low sugar only
                continue

        # B. Standard Safety Guardrail (Always on)
        if health_condition == "Hypertension" and row['sodium_mg'] > 500: continue

        final_suggestions.append(row)

    # 5. RANDOMIZATION (The Groundhog Day Fix)
    # If we found safe options, shuffle them and pick 3
    if not final_suggestions:
        print("⚠️ No options found (Strict Mode might be too strict!).")
    else:
        # Shuffle the list to ensure variety every day
        random.shuffle(final_suggestions)

        # Pick top 3 (or fewer if we don't have 3)
        daily_menu = final_suggestions[:3]

        print(f"✅ Found {len(final_suggestions)} options. Here are 3 distinct ones for today:\n")
        for row in daily_menu:
             print(f"🍲 {row.get('name', 'Unknown Dish')}")
             print(f"   📊 Sugar: {row.get('sugar_g')}g | Sodium: {row.get('sodium_mg')}mg")
             print("   -----------------")

In [ ]:
get_dynamic_recommendation("Hypertensive", "Lunch", data)


🔍 REQUEST: 'Lunch' for 'Hypertensive'
✅ Found 11 options. Here are 3 distinct ones for today:

🍲 Tubani (Steamed Bean Pudding)
   📊 Sugar: 1.0g | Sodium: 50mg
   -----------------
🍲 Kokonte with Palm Nut Soup
   📊 Sugar: 3.0g | Sodium: 410mg
   -----------------
🍲 Boiled Plantain and Kontomire Stew
   📊 Sugar: 12.0g | Sodium: 320mg
   -----------------


In [ ]:
get_dynamic_recommendation("Hypertensive", "Lunch", data, current_reading=150)


🔍 REQUEST: 'Lunch' for 'Hypertensive'
✅ Found 11 options. Here are 3 distinct ones for today:

🍲 Bulgur Wheat Jollof
   📊 Sugar: 2.5g | Sodium: 420mg
   -----------------
🍲 Boiled Plantain and Kontomire Stew
   📊 Sugar: 12.0g | Sodium: 320mg
   -----------------
🍲 Aboboi (Bambara Beans) with Tatale
   📊 Sugar: 28.0g | Sodium: 120mg
   -----------------
